### Train Pollinator Classifier (5-class)

Trains a single model — bumblebee / fly / butterfly / other / background — as an alternative to the two-stage approach.
> **Standalone only.** Not used by the Django backend or `colab/colab_master_pipeline.ipynb`. To run it at inference time, set `type: 'five_class'` in `PIPELINES` inside `infer_cropbased.ipynb`.

**Input** — `data/training/annotated_crops/{bumblebee,fly,butterfly,other,background}/`  
**Output** — `outputs/training/model_runs/{RUN_NAME}_{ts}/` · `models/5group_efficientnet.pth` and/or `models/5group_insectnet.pth`

**Must edit (Cell 2):**

| Variable | Risk if skipped |
|----------|-----------------|
| `TRAIN_MODEL` | `ValueError` if not `'efficientnet'`, `'insectnet'`, or `'both'` |
| `BASE_DIR` (Cell 1) | Everything fails — change only when running locally |

**Optional (Cell 2):** `EPOCHS_S1` (20), `EPOCHS_S2` (0 — skip recommended), `LR_S1` (1e-3), `BG_RATIO` (3), `BATCH` (32), `DATASETS` ([] = all sub-folders)

**Background sampling:** balanced across camera plots — each plot contributes equally. If crop filenames change, update only `parse_plot_key()` in Cell 3 — see its docstring.

**Note on false positives:** no binary gate means any non-background argmax = detection even at 1 % confidence. Use `CONF_THRESHOLDS` in `evaluate.ipynb` to filter at eval time.


##### Cell 1 — Environment  ← edit `BASE_DIR` for local runs
Sets all paths.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — ENVIRONMENT  ← only edit this cell
# ════════════════════════════════════════════════════════════
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    import zipfile, os
    DRIVE_ROOT = Path('/content/drive/MyDrive')
    ZIP_PATH   = DRIVE_ROOT / 'pollinator-colab.zip'
    EXTRACT_TO = Path('/content/pollinator-colab')
    if not EXTRACT_TO.exists():
        print(f'Extracting {ZIP_PATH.name} ...')
        with zipfile.ZipFile(ZIP_PATH) as z:
            z.extractall('/content/')
        print('✓ Extracted to /content/pollinator-colab')
    else:
        print('✓ Already extracted')
    BASE_DIR   = EXTRACT_TO
    DRIVE_BASE = DRIVE_ROOT / 'pollinator-colab'
else:
    # ← Edit this to your local repo root
    BASE_DIR   = Path('/path/to/pollinator-classification')
    DRIVE_BASE = BASE_DIR

MODEL_DIR   = BASE_DIR / 'models'
LABELED_DIR = BASE_DIR / 'data' / 'training' / 'annotated_crops'
INSECTNET_W = BASE_DIR / 'InsectNet' / 'model.pth'
WEB_IMG_DIR = BASE_DIR / 'data' / 'web_images'

# Training outputs go to local SSD on Colab (fast); saved to Drive after training.
LOCAL_TRAINING = Path('/content/outputs/training') if IN_COLAB else BASE_DIR / 'outputs' / 'training'
LOCAL_TRAINING.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Env        : {"Colab" if IN_COLAB else "Local"}')
print(f'BASE_DIR   : {BASE_DIR}  exists={BASE_DIR.exists()}')
print(f'MODEL_DIR  : {MODEL_DIR}  exists={MODEL_DIR.exists()}')
print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')
print(f'INSECTNET_W: {INSECTNET_W}  exists={INSECTNET_W.exists()}')
print(f'WEB_IMG_DIR: {WEB_IMG_DIR}  exists={WEB_IMG_DIR.exists()}')



##### Cell 2 — Config  ← **edit before training**
Set `TRAIN_MODEL`, epochs, LR, and batch size.

In [ ]:
# ── Which sub-datasets to use ──────────────────────────────────────────
# Sub-folder names inside data/training/annotated_crops/.
# Leave empty [] to use ALL sub-folders automatically (excluding 'progress').
DATASETS = []   # e.g. ['labeled_ls', 'labeled_mb']

RUN_NAME   = 'classifier_5class'  # ← label appended to timestamp

CLASSES_5 = ['bumblebee','fly','butterfly','other','background']

ALIAS_5 = {
    'bumblebee':'bumblebee', 'fly':'fly',
    'butterfly':'butterfly', 'butterfly_moth':'butterfly',
    'other':'other',         'background':'background',
}
WEB_ALIAS_5 = {
    'bumblebee':'bumblebee','fly':'fly',
    'butterfly':'butterfly','butterfly_moth':'butterfly','other':'other',
    # no background in web images
}

TRAIN_MODEL = 'both'  # 'efficientnet' | 'insectnet' | 'both'
IMG_SIZE    = 224
BATCH       = 32
EPOCHS_S1   = 20
EPOCHS_S2   = 0     # 0 = skip (recommended)
LR_S1       = 1e-3
LR_S2       = 1e-4
BG_RATIO    = 3
SEED        = 42
WEB_DIR     = None  # set to BASE_DIR/'web_images' if available

print(f'LABELED_DIR: {LABELED_DIR}  exists={LABELED_DIR.exists()}')


##### Cell 2b — Output paths
Builds the timestamped run directory. No edits needed.

In [ ]:
from datetime import datetime

_ts     = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR = BASE_DIR / 'outputs' / 'training' / 'model_runs' / f'{RUN_NAME}_{_ts}'
RUN_DIR.mkdir(parents=True, exist_ok=True)

import json as _json
_json.dump({
    'run_name': RUN_NAME, 'timestamp': _ts,
    'train_model': TRAIN_MODEL,
    'epochs_s1': EPOCHS_S1, 'epochs_s2': EPOCHS_S2,
    'lr_s1': LR_S1, 'lr_s2': LR_S2,
    'img_size': IMG_SIZE, 'batch': BATCH,
    'classes_5': CLASSES_5, 'bg_ratio': BG_RATIO,
}, open(RUN_DIR / 'config.json', 'w'), indent=2)

print(f'Run directory : {RUN_DIR}')
print('Checkpoints + curves saved here; best models also copied to models/')

# ── Resolve dataset sub-folders ─────────────────────────────────
if DATASETS:
    DATASET_DIRS = [LABELED_DIR / ds for ds in DATASETS]
else:
    DATASET_DIRS = sorted([d for d in LABELED_DIR.iterdir()
                           if d.is_dir() and d.name != 'progress'])
print(f'Datasets : {[d.name for d in DATASET_DIRS]}')


##### Cell 3 — Imports + training utilities
Loads PyTorch, model definitions, and helpers. Just run.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')


##### Cell 3b — Plotting utilities
Defines loss/accuracy curve helpers. Just run.

In [ ]:
import shutil, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import defaultdict
import numpy as np, torch, torch.nn as nn
import torchvision, torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
try:
    from sklearn.metrics import classification_report
    HAS_SKLEARN=True
except ImportError:
    HAS_SKLEARN=False

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

def letterbox(img, size):
    w,h=img.size; ms=max(w,h)
    sq=Image.new('RGB',(ms,ms),(0,0,0)); sq.paste(img,((ms-w)//2,(ms-h)//2))
    return sq.resize((size,size),Image.BILINEAR)

class CropDataset(Dataset):
    def __init__(self, samples, tf): self.s=samples; self.tf=tf
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        p,l=self.s[i]; return self.tf(Image.open(p).convert('RGB')), l

def make_tf(sz, aug=False):
    base=[T.Lambda(lambda i: letterbox(i,sz)), T.ToTensor(),
          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    if aug:
        base=[T.Lambda(lambda i: letterbox(i,sz)),
              T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
              T.ColorJitter(0.3,0.3,0.2,0.05)]+base[1:]
    return T.Compose(base)

def make_loader(samples, idxs, sz, batch, aug=False, weighted=True):
    sub=[samples[i] for i in idxs]; ds=CropDataset(sub,make_tf(sz,aug))
    if weighted and sub:
        labs=[s[1] for s in sub]; cnt=np.bincount(labs,minlength=max(labs)+1)
        wts=[1.0/max(1,cnt[l]) for l in labs]
        return DataLoader(ds,batch_size=batch,
                          sampler=WeightedRandomSampler(wts,len(wts)),num_workers=2,pin_memory=True)
    return DataLoader(ds,batch_size=batch,shuffle=False,num_workers=2,pin_memory=True)

def parse_plot_key(path):
    """
    Extract a per-camera-plot group key from a crop filename.

    ── WHY THIS EXISTS ──────────────────────────────────────────────────────
    Background crops are not distributed evenly: a camera at a busy flower
    plot may generate 10× more background crops than a quiet one.  Without
    plot-balanced sampling the binary classifier would be heavily biased
    toward the background texture of whichever plot had the most motion.
    This function maps every crop filename to the camera plot it came from
    so that sample_bg() can draw an equal quota from each plot.

    ── EXPECTED FILENAME FORMAT ─────────────────────────────────────────────
    Crops are saved by infer_cropbased.ipynb with names derived from the
    source image filename.  The current Arctic field-camera convention is:

      hdd_<n>_<year>_<site>_<species>_<plot>_[<date>_]<cam>_..._crop_<i>.jpg

    Examples:
      hdd_1_2025_cg_Vamy_p1_101_WSCT__WSCT3529_crop_9_normal_roi.jpg
      hdd_2_2025_desert_Asa_p2_20250729_102_WSCT__WSCT1266_crop_2.jpg

    Field positions in the underscore-split stem:
      parts[0] = 'hdd'          (hard-drive prefix — identifies this format)
      parts[1] = drive number   (1, 2, …)
      parts[2] = year           (2025, …)
      parts[3] = site           (cg, desert, …)
      parts[4] = species/area   (Vamy, Asa, …)
      parts[5] = plot           (p1, p2, p3, …)

    Returns: 'hdd<n>_<site>_<species>_<plot>'  e.g. 'hdd1_cg_Vamy_p1'

    ── HOW TO ADAPT FOR A NEW NAMING CONVENTION ─────────────────────────────
    If the crop filenames change in a future field season, update ONLY this
    function.  Keep the return value as a string that uniquely identifies
    one camera plot.  Everything else (sample_bg, training cells) stays the
    same.

    Examples of what you might return for other formats:
      'CAM01_plot3'     — if your files look like CAM01_plot3_0042.jpg
      path.parent.name  — if crops are already in per-plot sub-folders

    Falls back to 'unknown' (all crops go into one group) if the filename
    does not match the expected format — sampling still works but is no
    longer plot-balanced.
    """
    parts = Path(path).stem.split('_')
    try:
        if parts[0] == 'hdd' and len(parts) > 6:
            return f'hdd{parts[1]}_{parts[3]}_{parts[4]}_{parts[5]}'
    except IndexError:
        pass
    return 'unknown'


def sample_bg(bg_paths, n_total, seed=42):
    """
    Sample n_total background crops balanced across camera plots.

    Uses parse_plot_key() to group crops by plot, then gives each plot an
    equal quota so that no single busy plot can dominate the training set.

    Args:
        bg_paths : list of Path — all available background crop paths
        n_total  : int — how many to sample in total
        seed     : int — RNG seed for reproducibility

    Prints a per-group breakdown so you can verify the balance.
    If a plot has fewer crops than its quota, it contributes all it has
    (the total sampled may be slightly below n_total in that case).
    """
    if not n_total:
        return []

    groups = {}
    for p in bg_paths:
        groups.setdefault(parse_plot_key(p), []).append(p)

    rng = np.random.default_rng(seed)
    for imgs in groups.values():
        rng.shuffle(imgs)

    n_groups  = len(groups)
    quota     = n_total // n_groups
    remainder = n_total  % n_groups

    print(f'Background sampling: {n_groups} plot group(s), quota={quota}/group')
    for key, imgs in sorted(groups.items()):
        print(f'  {key:30}: {len(imgs):>5} available')

    sampled = []
    for i, key in enumerate(sorted(groups)):
        take = quota + (1 if i < remainder else 0)
        sampled.extend(groups[key][:min(take, len(groups[key]))])

    rng.shuffle(sampled)
    print(f'Sampled {len(sampled)} background crops (target {n_total})')
    return sampled



def stratified_split(samples, val_frac=0.2, test_frac=0.1, seed=42):
    """Split list of (path, label) into train/val/test indices, stratified by class."""
    rng = np.random.default_rng(seed)
    by_cls = defaultdict(list)
    for i, (_, l) in enumerate(samples): by_cls[l].append(i)
    tr, va, te = [], [], []
    for l, idxs in by_cls.items():
        rng.shuffle(idxs); n = len(idxs)
        nva = max(1, int(n*val_frac)); nte = max(1, int(n*test_frac))
        tr.extend(idxs[nva+nte:]); va.extend(idxs[nva:nva+nte]); te.extend(idxs[:nva])
    return tr, va, te



def collect_crops(dataset_dirs, classes, alias):
    """Collect (path, label_idx) tuples from sub-folders in each dataset dir."""
    ci = {c: i for i, c in enumerate(classes)}
    smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci:
                continue
            dd = Path(labeled_dir) / folder
            if not dd.exists():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                for p in dd.glob(ext):
                    smp.append((p, ci[cls]))
    counts = {i: 0 for i in range(len(classes))}
    for _, l in smp:
        counts[l] += 1
    return smp, counts


def build_efficientnet(n_classes):
    """EfficientNet-B2 pretrained on ImageNet, head replaced for n_classes."""
    print(f'Building EfficientNet-B2 (ImageNet weights, {n_classes} classes)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {n_train:,} trainable params (full fine-tune from ImageNet)')
    return model


def build_insectnet(weights_path, n_classes, unfreeze_last_block=False):
    """InsectNet backbone (RegNet-Y-32GF), frozen by default, head replaced for n_classes."""
    print(f'Building InsectNet ({n_classes} classes, RegNet-Y-32GF backbone)')
    if not Path(weights_path).exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'  Loading weights: {weights_path}')
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('  InsectNet weights loaded ✓')
    model.fc = nn.Linear(3712, n_classes)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    if unfreeze_last_block:
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    frozen_str = 'frozen backbone' if not unfreeze_last_block else 'last block + fc unfrozen'
    print(f'  Trainable: {n_train:,} / {n_total:,} params ({frozen_str})')
    return model


def _train_epoch(model, loader, optimizer, criterion, device):
    """Single training epoch; returns (avg_loss, accuracy)."""
    model.train()
    ls = cor = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        ls  += loss.item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
    return ls / tot, cor / tot


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    """Evaluate model; returns dict with loss, acc, macro_f1, per_f1, preds, labels."""
    model.eval()
    ls = cor = tot = 0
    ap, al = [], []
    tp = {i: 0 for i in range(len(classes))}
    fp = {i: 0 for i in range(len(classes))}
    fn = {i: 0 for i in range(len(classes))}
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        preds = out.argmax(1)
        ls  += criterion(out, labels).item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
        ap.extend(preds.cpu().tolist())
        al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i] += ((preds == i) & (labels == i)).sum().item()
            fp[i] += ((preds == i) & (labels != i)).sum().item()
            fn[i] += ((preds != i) & (labels == i)).sum().item()
    pf = {}; pr = {}
    for i in range(len(classes)):
        p2 = tp[i] / max(1, tp[i] + fp[i])
        r  = tp[i] / max(1, tp[i] + fn[i])
        pf[classes[i]] = 2 * p2 * r / max(1e-8, p2 + r)
        pr[classes[i]] = r
    # Combined recall across all non-background classes:
    # of all actual insects, what fraction was NOT missed as background?
    _ins_cls = [c for c in classes if c != 'background']
    _ins_idx = [i for i, c in enumerate(classes) if c != 'background']
    _ins_tp  = sum(tp[i] for i in _ins_idx)
    _ins_fn  = sum(fn[i] for i in _ins_idx)
    insect_recall = _ins_tp / max(1, _ins_tp + _ins_fn)
    return {'loss': ls / max(1, tot), 'acc': cor / max(1, tot),
            'macro_f1': sum(pf.values()) / max(1, len(classes)),
            'per_f1': pf, 'per_recall': pr,
            'insect_recall': insect_recall,
            'preds': ap, 'labels': al}


def run_training(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt,
                 device, classes, run_dir, in_colab,
                 best_metric='macro_f1'):
    """Full training loop with cosine LR schedule.

    Saves the best checkpoint by val metric (default macro_f1).
    For binary insect detection set best_metric='recall_insect' to avoid
    missing insects — recall of the insect class is printed each epoch.
    Returns the model loaded with the best weights.
    """
    w = torch.tensor([1.0 / max(1, counts.get(i, 1)) for i in range(len(classes))],
                     dtype=torch.float, device=device)
    crit  = nn.CrossEntropyLoss(weight=w / w.sum())
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_f1 = 0.0
    hist    = {'tr': [], 'va': [], 'f1': []}
    hdr = (f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}'
           f'  {"InsRec":>8}  {"Acc":>6}  '
           + '  '.join(f'{c[:8]:>9}' for c in classes))
    print(f'\n{"=" * 70}\n{name}  epochs={epochs}  lr={lr}  best={best_metric}\n{"=" * 70}\n{hdr}')

    for ep in range(1, epochs + 1):
        tl, ta = _train_epoch(model, tr_ldr, opt, crit, device)
        vr     = eval_epoch(model, va_ldr, crit, device, classes)
        sched.step()

        # Select best checkpoint by the chosen metric:
        #   'macro_f1'      — balanced across all classes (default, group classifier)
        #   'recall_insect' — insect-class recall (binary: don't miss insects)
        #   'recall_nonbg'  — combined recall of all non-background classes
        #                     (5-class model: don't call any real insect 'background')
        if best_metric == 'macro_f1':
            _score = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _score = vr['insect_recall']
        else:
            # 'recall_<classname>'  e.g. 'recall_insect'
            _cls = best_metric.split('_', 1)[1]
            _score = vr['per_recall'].get(_cls, vr['macro_f1'])
        new_best = _score > best_f1
        if new_best:
            best_f1 = _score
            torch.save({'state_dict': model.state_dict(), 'classes': classes,
                        'val_macro_f1': vr['macro_f1'], 'val_score': best_f1,
                        'best_metric': best_metric, 'epoch': ep}, ckpt)

        pf = vr['per_f1']
        if best_metric == 'macro_f1':
            _mscore = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _mscore = vr['insect_recall']
        else:
            _cls = best_metric.split('_', 1)[1]
            _mscore = vr['per_recall'].get(_cls, 0)
        print(f'{ep:>4}  {tl:>8.4f}  {vr["loss"]:>8.4f}  '
              f'{vr["macro_f1"]:>8.3f}  {_mscore:>8.3f}  {vr["acc"]:>6.3f}  '
              + '  '.join(f'{pf.get(c, 0):>9.3f}' for c in classes)
              + ('  *' if new_best else ''))
        hist['tr'].append(tl)
        hist['va'].append(vr['loss'])
        hist['f1'].append(vr['macro_f1'])

    print(f'\nBest val {best_metric}: {best_f1:.3f}  →  {ckpt}')

    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist['tr'], label='train'); ax1.plot(hist['va'], label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'], lw=2, label='macro F1')
    ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path = Path(run_dir) / f'{name}_curves.png'
    plt.savefig(curves_path, dpi=100); plt.close()
    print(f'Curves: {curves_path}')

    # Load best weights before returning
    model.load_state_dict(
        torch.load(ckpt, map_location=device, weights_only=False)['state_dict'])
    return model


def collect_crops(dataset_dirs, classes, alias):
    """Collect (path, label_idx) tuples from sub-folders in each dataset dir."""
    ci = {c: i for i, c in enumerate(classes)}
    smp = []
    for labeled_dir in dataset_dirs:
        for folder, cls in alias.items():
            if cls not in ci:
                continue
            dd = Path(labeled_dir) / folder
            if not dd.exists():
                continue
            for ext in ('*.jpg', '*.jpeg', '*.png'):
                for p in dd.glob(ext):
                    smp.append((p, ci[cls]))
    counts = {i: 0 for i in range(len(classes))}
    for _, l in smp:
        counts[l] += 1
    return smp, counts


def build_efficientnet(n_classes):
    """EfficientNet-B2 pretrained on ImageNet, head replaced for n_classes."""
    print(f'Building EfficientNet-B2 (ImageNet weights, {n_classes} classes)')
    model = torchvision.models.efficientnet_b2(weights='IMAGENET1K_V1')
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {n_train:,} trainable params (full fine-tune from ImageNet)')
    return model


def build_insectnet(weights_path, n_classes, unfreeze_last_block=False):
    """InsectNet backbone (RegNet-Y-32GF), frozen by default, head replaced for n_classes."""
    print(f'Building InsectNet ({n_classes} classes, RegNet-Y-32GF backbone)')
    if not Path(weights_path).exists():
        raise FileNotFoundError(f'InsectNet weights not found: {weights_path}')
    print(f'  Loading weights: {weights_path}')
    model = torchvision.models.regnet_y_32gf()
    model.fc = nn.Linear(3712, 2526)
    state = torch.load(weights_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model'] if 'model' in state else state, strict=True)
    print('  InsectNet weights loaded ✓')
    model.fc = nn.Linear(3712, n_classes)
    nn.init.xavier_uniform_(model.fc.weight)
    nn.init.zeros_(model.fc.bias)
    for name, p in model.named_parameters():
        p.requires_grad = name.startswith('fc.')
    if unfreeze_last_block:
        for name, p in model.named_parameters():
            if 'trunk_output.block4' in name or name.startswith('fc.'):
                p.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    frozen_str = 'frozen backbone' if not unfreeze_last_block else 'last block + fc unfrozen'
    print(f'  Trainable: {n_train:,} / {n_total:,} params ({frozen_str})')
    return model


def _train_epoch(model, loader, optimizer, criterion, device):
    """Single training epoch; returns (avg_loss, accuracy)."""
    model.train()
    ls = cor = tot = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        preds = out.argmax(1)
        ls  += loss.item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
    return ls / tot, cor / tot


@torch.no_grad()
def eval_epoch(model, loader, criterion, device, classes):
    """Evaluate model; returns dict with loss, acc, macro_f1, per_f1, preds, labels."""
    model.eval()
    ls = cor = tot = 0
    ap, al = [], []
    tp = {i: 0 for i in range(len(classes))}
    fp = {i: 0 for i in range(len(classes))}
    fn = {i: 0 for i in range(len(classes))}
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        preds = out.argmax(1)
        ls  += criterion(out, labels).item() * labels.size(0)
        cor += (preds == labels).sum().item()
        tot += labels.size(0)
        ap.extend(preds.cpu().tolist())
        al.extend(labels.cpu().tolist())
        for i in range(len(classes)):
            tp[i] += ((preds == i) & (labels == i)).sum().item()
            fp[i] += ((preds == i) & (labels != i)).sum().item()
            fn[i] += ((preds != i) & (labels == i)).sum().item()
    pf = {}; pr = {}
    for i in range(len(classes)):
        p2 = tp[i] / max(1, tp[i] + fp[i])
        r  = tp[i] / max(1, tp[i] + fn[i])
        pf[classes[i]] = 2 * p2 * r / max(1e-8, p2 + r)
        pr[classes[i]] = r
    # Combined recall across all non-background classes:
    # of all actual insects, what fraction was NOT missed as background?
    _ins_cls = [c for c in classes if c != 'background']
    _ins_idx = [i for i, c in enumerate(classes) if c != 'background']
    _ins_tp  = sum(tp[i] for i in _ins_idx)
    _ins_fn  = sum(fn[i] for i in _ins_idx)
    insect_recall = _ins_tp / max(1, _ins_tp + _ins_fn)
    return {'loss': ls / max(1, tot), 'acc': cor / max(1, tot),
            'macro_f1': sum(pf.values()) / max(1, len(classes)),
            'per_f1': pf, 'per_recall': pr,
            'insect_recall': insect_recall,
            'preds': ap, 'labels': al}


def run_training(model, name, tr_ldr, va_ldr, epochs, lr, counts, ckpt,
                 device, classes, run_dir, in_colab,
                 best_metric='macro_f1'):
    """Full training loop with cosine LR schedule.

    Saves the best checkpoint by val metric (default macro_f1).
    For binary insect detection set best_metric='recall_insect' to avoid
    missing insects — recall of the insect class is printed each epoch.
    Returns the model loaded with the best weights.
    """
    w = torch.tensor([1.0 / max(1, counts.get(i, 1)) for i in range(len(classes))],
                     dtype=torch.float, device=device)
    crit  = nn.CrossEntropyLoss(weight=w / w.sum())
    opt   = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_f1 = 0.0
    hist    = {'tr': [], 'va': [], 'f1': []}
    hdr = (f'{"Ep":>4}  {"TrLoss":>8}  {"VaLoss":>8}  {"MacroF1":>8}'
           f'  {"InsRec":>8}  {"Acc":>6}  '
           + '  '.join(f'{c[:8]:>9}' for c in classes))
    print(f'\n{"=" * 70}\n{name}  epochs={epochs}  lr={lr}  best={best_metric}\n{"=" * 70}\n{hdr}')

    for ep in range(1, epochs + 1):
        tl, ta = _train_epoch(model, tr_ldr, opt, crit, device)
        vr     = eval_epoch(model, va_ldr, crit, device, classes)
        sched.step()

        # Select best checkpoint by the chosen metric:
        #   'macro_f1'      — balanced across all classes (default, group classifier)
        #   'recall_insect' — insect-class recall (binary: don't miss insects)
        #   'recall_nonbg'  — combined recall of all non-background classes
        #                     (5-class model: don't call any real insect 'background')
        if best_metric == 'macro_f1':
            _score = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _score = vr['insect_recall']
        else:
            # 'recall_<classname>'  e.g. 'recall_insect'
            _cls = best_metric.split('_', 1)[1]
            _score = vr['per_recall'].get(_cls, vr['macro_f1'])
        new_best = _score > best_f1
        if new_best:
            best_f1 = _score
            torch.save({'state_dict': model.state_dict(), 'classes': classes,
                        'val_macro_f1': vr['macro_f1'], 'val_score': best_f1,
                        'best_metric': best_metric, 'epoch': ep}, ckpt)

        pf = vr['per_f1']
        if best_metric == 'macro_f1':
            _mscore = vr['macro_f1']
        elif best_metric == 'recall_nonbg':
            _mscore = vr['insect_recall']
        else:
            _cls = best_metric.split('_', 1)[1]
            _mscore = vr['per_recall'].get(_cls, 0)
        print(f'{ep:>4}  {tl:>8.4f}  {vr["loss"]:>8.4f}  '
              f'{vr["macro_f1"]:>8.3f}  {_mscore:>8.3f}  {vr["acc"]:>6.3f}  '
              + '  '.join(f'{pf.get(c, 0):>9.3f}' for c in classes)
              + ('  *' if new_best else ''))
        hist['tr'].append(tl)
        hist['va'].append(vr['loss'])
        hist['f1'].append(vr['macro_f1'])

    print(f'\nBest val {best_metric}: {best_f1:.3f}  →  {ckpt}')

    # Plot training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(hist['tr'], label='train'); ax1.plot(hist['va'], label='val')
    ax1.set_title('Loss'); ax1.legend(); ax1.grid(True)
    ax2.plot(hist['f1'], lw=2, label='macro F1')
    ax2.set_title('Macro F1'); ax2.grid(True)
    plt.suptitle(name); plt.tight_layout()
    curves_path = Path(run_dir) / f'{name}_curves.png'
    plt.savefig(curves_path, dpi=100); plt.close()
    print(f'Curves: {curves_path}')

    # Load best weights before returning
    model.load_state_dict(
        torch.load(ckpt, map_location=device, weights_only=False)['state_dict'])
    return model

print('Training utilities loaded.')


##### Cell 4 — Collect and split data

Loads annotated crops (5 classes including background) and builds stratified train/val/test splits.

**Background sampling** is balanced across camera plots using `parse_plot_key()` + `sample_bg()`:
each plot contributes an equal quota so a busy plot cannot dominate the training set.

> **If your crop filenames change** (new field season, different camera naming), update
> `parse_plot_key()` in Cell 3 — that is the only function you need to touch.
> See its docstring for examples of other naming formats.


In [ ]:
arctic_smp, counts_arc = collect_crops(DATASET_DIRS, CLASSES_5, ALIAS_5)

# Balance background
n_ins = sum(counts_arc[i] for i in range(4))
bg_all = [p for p,l in arctic_smp if l==4]
bg_bal = sample_bg(bg_all, n_ins*BG_RATIO, SEED)
arctic_smp = [(p,l) for p,l in arctic_smp if l!=4] + [(p,4) for p in bg_bal]
counts_arc[4] = len(bg_bal)

from pathlib import Path
web_smp, counts_web = (collect_crops([WEB_DIR], CLASSES_5, WEB_ALIAS_5)
                       if WEB_DIR and Path(WEB_DIR).exists() else ([],{i:0 for i in range(5)}))
arc_tr,arc_va,arc_te = stratified_split(arctic_smp, seed=SEED)
web_tr,web_va,web_te = stratified_split(web_smp, seed=SEED) if web_smp else ([],[],[])
print(f'Arctic: {len(arctic_smp)}  Web: {len(web_smp)}')


##### Cell 5 — Train 5-class classifiers
Trains EfficientNet and/or InsectNet. Saves best weights to `models/5group_*.pth`.

In [ ]:
import shutil

def train_5class(backbone):
    print(f'\n{"#"*60}\n# 5-class {backbone.upper()}\n{"#"*60}')
    model = (build_efficientnet(5) if backbone=='efficientnet'
             else build_insectnet(INSECTNET_W, 5)).to(DEVICE)
    if web_smp:
        combined = arctic_smp+web_smp
        comb_tr  = arc_tr+[len(arctic_smp)+i for i in web_tr]
        comb_cnt = {i:counts_arc.get(i,0)+counts_web.get(i,0) for i in range(5)}
        tr_s1 = make_loader(combined, comb_tr, IMG_SIZE, BATCH, aug=True)
        va_s1 = make_loader(arctic_smp+web_smp,
                            arc_va+[len(arctic_smp)+i for i in web_va], IMG_SIZE, BATCH)
    else:
        comb_cnt=counts_arc; tr_s1=make_loader(arctic_smp,arc_tr,IMG_SIZE,BATCH,aug=True)
        va_s1=make_loader(arctic_smp,arc_va,IMG_SIZE,BATCH)
    ckpt_s1 = RUN_DIR/f'5group_{backbone}_stage1.pth'
    ckpt_f  = RUN_DIR/f'5group_{backbone}.pth'
    model = run_training(model,f'5class_{backbone}_s1',tr_s1,va_s1,
                         EPOCHS_S1,LR_S1,comb_cnt,ckpt_s1,DEVICE,CLASSES_5,RUN_DIR,IN_COLAB,
                         best_metric='recall_nonbg')
    if EPOCHS_S2==0:
        print('Stage2=0 — copying Stage 1 as final.'); shutil.copy(ckpt_s1,ckpt_f)
    else:
        model.load_state_dict(
            __import__('torch').load(ckpt_s1,map_location=DEVICE,weights_only=False)['state_dict'])
        tr_s2=make_loader(arctic_smp,arc_tr,IMG_SIZE,BATCH,aug=True)
        va_s2=make_loader(arctic_smp,arc_va,IMG_SIZE,BATCH)
        model=run_training(model,f'5class_{backbone}_s2',tr_s2,va_s2,
                           EPOCHS_S2,LR_S2,counts_arc,ckpt_f,DEVICE,CLASSES_5,RUN_DIR,IN_COLAB)
    model.load_state_dict(
        __import__('torch').load(ckpt_f,map_location=DEVICE,weights_only=False)['state_dict'])
    te_ldr=make_loader(arctic_smp,arc_te,IMG_SIZE,BATCH)
    ta=eval_epoch(model,te_ldr,__import__('torch').nn.CrossEntropyLoss(),DEVICE,CLASSES_5)
    print(f'Test Arctic: MacroF1={ta["macro_f1"]:.3f}  Acc={ta["acc"]:.3f}')
    if HAS_SKLEARN:
        print(classification_report(ta['labels'],ta['preds'],target_names=CLASSES_5,digits=3))
    # Copy to models/ for inference
    import shutil as _sh
    _sh.copy(ckpt_f, MODEL_DIR / f'5group_{backbone}.pth')
    print(f'  → models/5group_{backbone}.pth updated')
    # Save report
    if HAS_SKLEARN:
        rpt = classification_report(ta['labels'], ta['preds'],
                                    target_names=CLASSES_5, digits=3)
        (RUN_DIR / f'5class_{backbone}_test_report.txt').write_text(
            f'5group_{backbone}  test_macro_f1={ta["macro_f1"]:.3f}\n\n' + rpt)
        print(f'  → 5class_{backbone}_test_report.txt saved')

    if HAS_SKLEARN:
        import matplotlib.pyplot as _plt
        from sklearn.metrics import confusion_matrix as _cm
        _cm_arr = _cm(ta['labels'], ta['preds'])
        _fig, _ax = _plt.subplots(figsize=(6, 5))
        _im = _ax.imshow(_cm_arr, interpolation='nearest', cmap='Blues')
        _plt.colorbar(_im, ax=_ax)
        _ax.set_xticks(range(len(CLASSES_5)))
        _ax.set_yticks(range(len(CLASSES_5)))
        _ax.set_xticklabels([f'Pred: {c}' for c in CLASSES_5], fontsize=9, rotation=20, ha='right')
        _ax.set_yticklabels([f'True: {c}' for c in CLASSES_5], fontsize=9)
        for _r in range(len(CLASSES_5)):
            for _c in range(len(CLASSES_5)):
                _color = 'white' if _cm_arr[_r, _c] > _cm_arr.max() / 2 else 'black'
                _ax.text(_c, _r, str(_cm_arr[_r, _c]), ha='center', va='center',
                         fontsize=11, fontweight='bold', color=_color)
        _ax.set_title(f'5group_{backbone} — Confusion Matrix (test set)', fontsize=11)
        _plt.tight_layout()
        _cm_path = RUN_DIR / f'5class_{backbone}_confusion_matrix.png'
        _plt.savefig(_cm_path, dpi=120); _plt.close()
        print(f'  → 5class_{backbone}_confusion_matrix.png saved')
    import json as _json
    _ckpt_meta = __import__('torch').load(ckpt_final, map_location='cpu', weights_only=False)
    _results = {
        'model': f'5group_{backbone}',
        'best_epoch': int(_ckpt_meta.get('epoch', -1)),
        'val_macro_f1': float(_ckpt_meta.get('val_macro_f1', 0)),
        'test_macro_f1': float(ta['macro_f1']),
        'test_acc': float(ta['acc']),
        'test_per_f1': {k: float(v) for k, v in ta.get('per_f1', {}).items()},
        'test_per_recall': {k: float(v) for k, v in ta.get('per_recall', {}).items()},
    }
    (RUN_DIR / f'5class_{backbone}_results.json').write_text(_json.dumps(_results, indent=2))
    print(f'  → 5class_{backbone}_results.json saved')
    return ta

res={}
if TRAIN_MODEL in ('efficientnet','both'): res['efficientnet']=train_5class('efficientnet')
if TRAIN_MODEL in ('insectnet','both'):    res['insectnet']=train_5class('insectnet')
if len(res)>1:
    print('\n=== COMPARISON ===')
    for name,r in res.items(): print(f'{name:15}: F1={r["macro_f1"]:.3f}')

print(f'\nAll outputs in: {RUN_DIR}')
